# 🎮 HOA Warfare — Voice Regeneration Studio

Regenerates all **1,451 voice lines** across **20 factions** using free, natural-sounding neural voices (Microsoft Edge TTS — free, unlimited, no API key).

**What this does:**
1. Downloads the voice script from the public repo
2. Assigns each faction a distinct, character-matched voice (see casting table below)
3. Generates every line → `.ogg` files + `manifest.json` (same format the game already uses)
4. Packages everything as a `.zip` you download and drop into `web/vo/`

**Phase 2** (at the bottom): voice *cloning* with XTTS v2 — drop in 10-30 sec reference clips per faction for even more character-accurate voices.

⏱️ Runtime: ~20-40 min on free Colab CPU (resumable — re-running skips finished lines).

## Step 1 — Install

In [ ]:
!pip install -q edge-tts
!ffmpeg -version 2>&1 | head -1

## Step 2 — Download the voice script

In [ ]:
import json, urllib.request

URL = "https://raw.githubusercontent.com/bparlette/hoa-warfare/main/data/hoa_content.json"
urllib.request.urlretrieve(URL, "/tmp/hoa_content.json")

with open("/tmp/hoa_content.json") as f:
    CONTENT = json.load(f)

factions = CONTENT["factions"]
print(f"{len(factions)} factions loaded")

# Flatten to (line_id, faction, event, text) rows
rows = []
for fid, fdata in factions.items():
    for event, variants in fdata.get("lines", {}).items():
        for variant_name, lines in variants.items():
            vcode = "c" if variant_name == "clean" else "s"
            for i, text in enumerate(lines):
                lid = f"{fid}.{event}.{vcode}.{i}"
                rows.append((lid, fid, event, text))
print(f"{len(rows)} voice lines total")

## Step 3 — Voice casting 🎭

Each faction gets a **distinct** neural voice matched to its personality. No pitch-shifting tricks — these are naturally different voices.

| Faction | Voice | Why |
|---|---|---|
| Karens | en-US-AvaNeural (f, pitch+) | Sharp, clear, complaint-ready |
| Valley Girls | en-US-JennyNeural (f, fast) | Bubbly, uptalk energy |
| Theater Kids | en-GB-SoniaNeural (f) | Dramatic British theatre |
| Woke Libs | en-US-MichelleNeural (f) | Earnest, precise |
| MLM Huns | en-US-AriaNeural (f, fast) | Warm, salesy |
| Tradwives | en-US-SaraNeural (f, slow) | Soft, serene |
| Frat House | en-US-AndrewNeural (m) | Young bro |
| Brain Rot | en-US-JasonNeural (m, fast) | Hyperactive zoomer |
| Old Money | en-GB-RyanNeural (m, slow) | Bored aristocrat |
| Rednecks | en-US-DavisNeural (m) | Gravelly, aggrieved |
| Gym Rats | en-US-GuyNeural (m) | Deep, breathy |
| Clout Streamers | en-US-BrandonNeural (m, fast) | Hype streamer |
| VC Bros | en-US-ChristopherNeural (m) | Smooth operator |
| HOA Dads | en-US-RogerNeural (m) | Clipboard authority |
| Emo Kids | en-US-BrianNeural (m, slow, pitch-) | Deadpan |
| The Gays | en-AU-WilliamNeural (m) | Flamboyant, precise |
| Trumpers | en-US-SteffanNeural (m) | Blustery |
| Furries | en-GB-ThomasNeural (m) | Earnest, excitable |
| Weeb Mods | en-US-EricNeural (m, fast) | Nasally mod |
| Pickleball Retirees | en-AU-NeilNeural (m, slow) | Raspy old-timer |

Edit the `CAST` dict below to swap any voice — [full voice list](https://speech.platform.bing.com/consumer/speech/synthesizers/voices/list?trustedscope=MyEdge).

In [ ]:
# faction -> (edge voice id, rate %, pitch Hz)
# rate: "+25%" = 1.25x speed | pitch: "+10Hz" raises slightly
CAST = {
    # --- women ---
    "karens":         ("en-US-AvaNeural",   "+25%", "+15Hz"),
    "valleyGirls":    ("en-US-JennyNeural",    "+30%", "+10Hz"),
    "theaterKids":    ("en-GB-SoniaNeural",    "+25%",  "+0Hz"),
    "wokeLibs":       ("en-US-MichelleNeural", "+25%",  "+0Hz"),
    "announcer":      ("en-US-GuyNeural",    "+10%",  "+0Hz"),  # PA/megaphone: authoritative suburban announcer
    "announcer":      ("en-US-GuyNeural",    "+10%",  "+0Hz"),  # PA/megaphone: authoritative suburban announcer
    "mlmHuns":        ("en-US-AriaNeural",     "+30%",  "+10Hz"),
    "tradwives":      ("en-US-SaraNeural",     "+15%",  "+0Hz"),
    # --- men ---
    "fratHouse":      ("en-US-AndrewNeural",     "+25%",  "+0Hz"),
    "brainRot":       ("en-US-JasonNeural",    "+40%", "+20Hz"),
    "oldMoney":       ("en-GB-RyanNeural",     "+10%", "-10Hz"),
    "rednecks":       ("en-US-DavisNeural",    "+25%",  "+0Hz"),
    "gymRats":        ("en-US-GuyNeural",      "+25%",  "+0Hz"),
    "cloutStreamers": ("en-US-BrandonNeural",   "+35%",  "+5Hz"),
    "wallstreetBets":         ("en-US-ChristopherNeural","+25%",  "+0Hz"),
    "hoaDads":        ("en-US-RogerNeural",    "+25%",  "+0Hz"),
    "emoKids":        ("en-US-BrianNeural",    "+10%", "-15Hz"),
    "fabulous":        ("en-AU-WilliamNeural",  "+25%",  "+5Hz"),
    "trumpers":       ("en-US-SteffanNeural",  "+25%",  "+0Hz"),
    "furries":        ("en-GB-ThomasNeural",   "+30%", "+10Hz"),
    "animeWeebs":     ("en-US-EricNeural",     "+35%", "+15Hz"),
    "pickleballRetirees": ("en-AU-NeilNeural",  "+15%", "-10Hz"),
}

missing = set(factions) - set(CAST)
assert not missing, f"No voice cast for: {missing}"
print(f"✅ {len(CAST)} factions cast, all distinct voices")

## Step 4 — Preview (optional)

Generate one line per faction so you can hear the casting before committing to the full run.

In [ ]:
import asyncio, edge_tts, os

os.makedirs("/tmp/vo_preview", exist_ok=True)

async def preview():
    for fid in CAST:
        # first kill line as the audition piece
        sample = next((r for r in rows if r[1] == fid and r[2] == "kill"), rows[0])
        lid, _, _, text = sample
        voice, rate, pitch = CAST[fid]
        out = f"/tmp/vo_preview/{fid}.mp3"
        if os.path.exists(out):
            continue
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        await comm.save(out)
        print(f"{fid:20s} ← {text[:50]}")

await preview()
print("\n🔊 Play files in /tmp/vo_preview/ from the Colab file browser")

## Step 5 — Generate all lines (the big run)

Resumable: skips any `.ogg` that already exists and passes validation.

In [ ]:
import asyncio, edge_tts, os, subprocess, json
from pathlib import Path

OUT = Path("/tmp/vo_out")
OUT.mkdir(exist_ok=True)

def valid_ogg(p):
    if not p.exists() or p.stat().st_size < 500:
        return False
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries",
                        "format=duration", "-of", "csv=p=0", str(p)],
                       capture_output=True, text=True)
    try:
        return float(r.stdout.strip()) > 0.2
    except ValueError:
        return False

def ogg_path(lid, fid, event, text):
    # fratHouse.kill.c.0 -> fratHouse_kill_c_0.ogg (matches existing convention)
    return OUT / (lid.replace(".", "_") + ".ogg")

async def synth_one(lid, fid, event, text):
    ogg = ogg_path(lid, fid, event, text)
    if valid_ogg(ogg):
        return ("skip", lid)
    voice, rate, pitch = CAST[fid]
    mp3 = OUT / (lid.replace(".", "_") + ".mp3")
    try:
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        await comm.save(str(mp3))
        # mp3 -> ogg (vorbis, game-ready)
        subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(mp3),
                        "-c:a", "libvorbis", "-q:a", "4", str(ogg)], check=True)
        mp3.unlink()
        return ("ok", lid)
    except Exception as e:
        return ("fail", f"{lid}: {e}")

async def run_all(batch=8):
    done, failed, skipped = 0, [], 0
    for i in range(0, len(rows), batch):
        chunk = rows[i:i+batch]
        results = await asyncio.gather(*[synth_one(*r) for r in chunk])
        for status, info in results:
            if status == "ok": done += 1
            elif status == "skip": skipped += 1
            else: failed.append(info)
        if (i // batch) % 10 == 0:
            print(f"  {i+len(chunk)}/{len(rows)} — done={done} skipped={skipped} failed={len(failed)}")
    return done, skipped, failed

done, skipped, failed = await run_all()
print(f"\n✅ done={done} skipped={skipped} failed={len(failed)}")
for f in failed[:10]:
    print("  FAIL:", f)

## Step 6 — Build manifest.json

Same schema the game already reads (`web/vo/manifest.json`).

In [ ]:
import subprocess, datetime

def probe_duration(p):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries",
                        "format=duration", "-of", "csv=p=0", str(p)],
                       capture_output=True, text=True)
    return round(float(r.stdout.strip()), 6)

def probe_peak(p):
    r = subprocess.run(["ffmpeg", "-i", str(p), "-af", "volumedetect",
                        "-f", "null", "/dev/null"],
                       capture_output=True, text=True, stderr=subprocess.STDOUT)
    for line in r.stdout.splitlines():
        if "max_volume" in line:
            v = line.split("max_volume:")[1].strip().split()[0]
            return float(v) if v != "-inf" else -60.0
    return -20.0

manifest = {"version": 2, "generated": datetime.date.today().isoformat(), "lines": {}}
for lid, fid, event, text in rows:
    ogg = ogg_path(lid, fid, event, text)
    if not ogg.exists():
        print("MISSING:", lid)
        continue
    voice, rate, pitch = CAST[fid]
    manifest["lines"][lid] = {
        "file": ogg.name,
        "duration": probe_duration(ogg),
        "faction": fid,
        "event": event,
        "text": text,
        "voice": voice,
        "pitch": pitch,
        "rate": rate,
        "peak_db": round(probe_peak(ogg), 1),
    }

manifest["count"] = len(manifest["lines"])
with open(OUT / "manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f"✅ manifest.json — {manifest['count']} lines")

## Step 7 — Package & download

In [ ]:
!cd /tmp/vo_out && rm -f *.mp3 && zip -q -r /tmp/hoa_warfare_voices.zip *.ogg manifest.json && ls -lh /tmp/hoa_warfare_voices.zip

from google.colab import files
files.download("/tmp/hoa_warfare_voices.zip")

print("\n📦 Drop the .ogg files + manifest.json into web/vo/ in the repo")

---

# 🎭 Phase 2 — Voice CLONING with XTTS v2 (optional upgrade)

Edge TTS gives you 20 distinct, natural voices for free. If you want voices that match the characters *even more precisely* (a truly nasally Karen, a truly booming HOA Dad), clone from reference clips:
",
"> ⚠️ **Legal note:** cloning celebrity/character voices (FakeYou-style impressions) for a commercial game carries right-of-publicity risk — even via XTTS. Use original voices, voice actors, or commercially-licensed library voices for anything you sell. This Phase 2 is fine for parody/prototype use; get proper licensing before shipping cloned likenesses.

1. **Get reference audio** — 10-30 seconds per faction, single speaker, no background music. Have an AI find samples (YouTube clips, voice demos, etc.) or record them.
2. **Upload** the 20 clips to `/tmp/refs/` named `{faction}.wav` (e.g. `karens.wav`, `hoaDads.wav`).
3. **Run the cells below** — XTTS v2 clones each reference and regenerates all lines.

Needs a Colab GPU (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Phase 2 setup — only run if you have reference clips + GPU runtime
!pip install -q TTS
!ls /tmp/refs/ 2>/dev/null || echo "⚠️ No /tmp/refs/ yet — upload reference clips first"

In [ ]:
# Phase 2 generation — XTTS v2 zero-shot cloning
import torch, os
from TTS.api import TTS
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
assert device == "cuda", "Switch to GPU runtime for XTTS"

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

REFS = Path("/tmp/refs")
OUT2 = Path("/tmp/vo_cloned")
OUT2.mkdir(exist_ok=True)

missing = [fid for fid in CAST if not (REFS / f"{fid}.wav").exists()]
assert not missing, f"Missing reference clips: {missing}"

for lid, fid, event, text in rows:
    ogg = OUT2 / (lid.replace(".", "_") + ".ogg")
    if ogg.exists() and ogg.stat().st_size > 500:
        continue
    wav_tmp = OUT2 / (lid.replace(".", "_") + ".wav")
    tts.tts_to_file(text=text,
                    speaker_wav=str(REFS / f"{fid}.wav"),
                    language="en",
                    file_path=str(wav_tmp))
    os.system(f"ffmpeg -y -v error -i {wav_tmp} -c:a libvorbis -q:a 4 {ogg} && rm {wav_tmp}")

print(f"✅ cloned voices in {OUT2}")
# Re-run Step 6 (manifest) pointed at OUT2, then Step 7 to package.